# Mapping unique isoform IDs to their unique canonical IDs

In this Notebook we will use the unique dataframes created in the last notebook to create a unique mapping from unique isoform IDs to their respective unique canonical ID.

# Setup
## Import Packages

In [63]:
import sys
import os
import session_info

# Add the '0_functions' folder to sys.path
sys.path.append(os.path.join(os.getcwd(), '..', '00_functions'))

In [64]:
import pandas as pd
import numpy as np
from functions import read_fastafile
import re

In [65]:
# Display session information
session_info.show()

## Set working directory

In [66]:
cwd = os.getcwd()
if not cwd.endswith("Isoform_Database/Jupyter_environment/01_UniProt"):
    print("WARNING: The working directory is not set to the '01_UniProt'!")
    print("Current working directory:", cwd)
else:
    print(f"Working directory is correctly set to the '01_Uniprot' folder (\"{cwd}\").")

Current working directory: /Users/lauranieba/Documents/Uni/Master_ETH/FS26/Lab_Rotation_Davos/Isoform_Database/Jupyter_environment/Isoform_Database_SIAF/01_UniProt


In [67]:
# Data directories
raw_data_dir = "data/raw_data"
mapping_dir = "data/mapping"
unique_dir = "data/unique"
duplicates_dir = "data/sequence_duplicates"

## Read in files
To create this mapping I have downloaded all canonical entries of the human proteome as a tsv instead of a fasta file from UniProt (date: 13.04.2026) to get a  dataframe with the canonical IDs of the human proteome and a column "Alternative Products" that contains the isoforms of each canonical sequence if there are any. The unique_canonical_mapping was created by hand and includes all the new unique identifiers for the duplicated sequences and the isoforms that are mapped to these canonical identifiers.

In [68]:
isoform_canonical_mapping = pd.read_csv(os.path.join(mapping_dir, 'uniprotkb_canonical_for_mapping_2026_04_13.tsv'), sep='\t')

canonical_unique = pd.read_csv(os.path.join(unique_dir, 'canonical_unique.csv'))
isoform_unique = pd.read_csv(os.path.join(unique_dir, 'isoform_unique.csv'))
full_proteome_unique = pd.read_csv(os.path.join(unique_dir, 'full_proteome_unique.csv'))

duplicates_list = pd.read_csv(os.path.join(duplicates_dir, 'duplicates_list_01042026.csv'))

## Get isoforms to canonical uniprot IDs
This function returns a list of caonical IDs and a list of the isoforms that can be merged into a dataframe. From the resulting dataframe I removed the IDs that belong to the duplicated sequences and added the unique IDs with the corresponding isoforms that will be mapped to these unique canonical IDs. The one unique isoforms ID will be mapped to both canonical identifiers, since we can't know which is the canonical ID.

In [69]:
def extract_isoform_split(text):
    if pd.isna(text) or not isinstance(text, str):
        return np.nan, [] # Change None to np.nan

    all_isoids = []
    for i in re.findall(r'IsoId=([\w-]+)', text):
        if i not in all_isoids:
            all_isoids.append(i)
    
    canonical_match = re.search(r'IsoId=([\w-]+);.*?Sequence=Displayed', text)
    # Change None to np.nan here as well
    canonical_id = canonical_match.group(1) if canonical_match else np.nan

    alternatives = [i for i in all_isoids if i != canonical_id]

    return canonical_id, alternatives

# Apply the function to create two new columns at once
iso_data = isoform_canonical_mapping['Alternative products (isoforms)'].apply(extract_isoform_split)

# Expand the tuple into two distinct columns
iso_canonical_clean = pd.DataFrame({
    'Entry': isoform_canonical_mapping['Entry'],
    'Canonical_Isoform_ID': iso_data.apply(lambda x: x[0]),
    'Isoforms': iso_data.apply(lambda x: x[1])
})

In [70]:
iso_canonical_clean

,Entry,Canonical_Isoform_ID,Isoforms
0,A0A087X1C5,NaN,[]
1,A0A096LP01,A0A096LP01-1,[A0A096LP01-2]
2,A0A0B4J2F0,NaN,[]
3,A0A0C5B5G6,NaN,[]
4,A0A0K2S4Q6,A0A0K2S4Q6-1,[A0A0K2S4Q6-2]
...,...,...,...
20426,Q9UI54,NaN,[]
20427,Q9UI72,NaN,[]
20428,Q9Y3F1,NaN,[]
20429,Q9Y4M8,NaN,[]


In [71]:
# remove IDs from duplicated seqeunces
duplicate_ids = duplicates_list["ID"].tolist()

duplicate_ids.append('P0DPK4')
duplicate_ids.append('Q96PT3')
duplicate_ids.append('Q9NZJ9')

duplicate_ids.remove('P0DPK4-2')
duplicate_ids.remove('Q96PT3-2')
duplicate_ids.remove('Q9NZJ9-2')


iso_canonical_no_duplicates = iso_canonical_clean[~iso_canonical_clean['Entry'].isin(duplicate_ids)]

In [72]:
#put unique IDs and isoforms in the a list of dictionaries to have a mapping of all isoforms in the human proteome

unique_mapping_set = [
    {"Entry": "P86791_P86790", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "P0DMW4_P0DMW5", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "P0DP23_P0DP24_P0DP25", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "P0CW20_P0CW19", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "P0DMV8_P0DMV9", "Canonical_Isoform_ID": "P0DMV8_P0DMV9", "Isoforms": ["P0DMV8-2"]},
    {"Entry": "P0DPH9_A0A1B0GTR3", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "O43812_Q96PT3", "Canonical_Isoform_ID": "O43812_Q96PT3", "Isoforms": ["Q96PT3-2"]},
    {"Entry": "P0CJ85_P0CJ86_P0CJ88_P0CJ89_P0CJ90", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "P0CJ75_P0DMP1", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "P04001_P0DN77_P0DN78", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "B7ZW38_P0DMR1", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "Q7Z3S9_P0DPK4", "Canonical_Isoform_ID": "Q7Z3S9_P0DPK4", "Isoforms": ["Q7Z3S9-2", "Q7Z3S9-3", "P0DPK4-2"]},
    {"Entry": "P0DPP9_Q3ZM63", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "A0A0B4J2D9_P0DP09", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "P01597_P04432", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "P01593_P01594", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "A0A075B6S9_P0DSN7", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "B0FP48_E5RIL1", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "Q8IZP1_Q6IPX1", "Canonical_Isoform_ID": "Q8IZP1_Q6IPX1", "Isoforms": ["Q8IZP1-2"]},
    {"Entry": "P86479_P86480_P86478_P86481_P86496", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "P01768_P0DP03", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "P0DMM9_P0DMN0", "Canonical_Isoform_ID": "P0DMM9_P0DMN0", "Isoforms": ["P0DMM9-2", "P0DMM9-3"]},
    {"Entry": "A0A1W2PP81_P0DW11_P0DW12_P0DW13_P0DW14", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "A0A0B4J1W7_P0DXC3", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "E9PJI5_P0DM63", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "A0A494C0Z2_P0DUD3_P0DUD4", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "A0A494C191_P0DTA3_P0DUD1_P0DUD2_P0DUX0", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "P62684_Q9YNA8", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "A0A0U1RR11_P0DPI3", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "A0A1B0GUQ0_A0A1B0GV22", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "A0A1B0GTK5_P0DP71", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "P0DTE7_P0DTE8_P0DUB6", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "P0DP73_P0DP74", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "P0DUQ2_P0DUQ1", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "A6NGN4_H0Y7S4", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "B3EWG3_B3EWG5_B3EWG6", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "P0DV73_P0DV74", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "P0DV75_P0DV76", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "A0A024RBG1_Q9NZJ9", "Canonical_Isoform_ID": "A0A024RBG1_Q9NZJ9", "Isoforms": ["Q9NZJ9-2", "Q9NZJ9-3"]},
    {"Entry": "Q69383_P61572_P61573_P61574_P61579", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "P0DPH7_P0DPH8", "Canonical_Isoform_ID": "P0DPH7_P0DPH8", "Isoforms": ["P0DPH7-2"]},
    {"Entry": "Q30KR1_P0DY27", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "P01614_A0A087WW87", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "P01615_A0A075B6P5", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "P0CV98_P0CW01", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "P0CG34_P0CG35_P0DX04", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "B7ZAQ6_P0CG08", "Canonical_Isoform_ID": "B7ZAQ6_P0CG08", "Isoforms": ["B7ZAQ6-2", "B7ZAQ6-3", "B7ZAQ6-4"]},
    {"Entry": "P0DI81_P0DI82", "Canonical_Isoform_ID": "P0DI81_P0DI82", "Isoforms": ["P0DI81-2", "P0DI81-3"]},
    {"Entry": "A0A0J9YXY3_P0DPF7", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "O76087_P0CL82_P0CL80_P0CL81", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "P0DMU7_P0DMU8_P0DMV0", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "Q5DJT8_P0DMV1_P0DMV2", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "Q9BYR9_P0C7H8", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "A0A0A6YYL3_H3BUK9", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "A0A075B759_P0DN26", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
    {"Entry": "E9PI22_P0DMB1", "Canonical_Isoform_ID": np.nan, "Isoforms": []},
]

# convert to dataframe
unique_mapping_df = pd.DataFrame(unique_mapping_set)
unique_mapping_df

,Entry,Canonical_Isoform_ID,Isoforms
0,P86791_P86790,NaN,[]
1,P0DMW4_P0DMW5,NaN,[]
2,P0DP23_P0DP24_P0DP25,NaN,[]
3,P0CW20_P0CW19,NaN,[]
4,P0DMV8_P0DMV9,P0DMV8_P0DMV9,[P0DMV8-2]
5,P0DPH9_A0A1B0GTR3,NaN,[]
6,O43812_Q96PT3,O43812_Q96PT3,[Q96PT3-2]
7,P0CJ85_P0CJ86_P0CJ88_P0CJ89_P0CJ90,NaN,[]
8,P0CJ75_P0DMP1,NaN,[]
9,P04001_P0DN77_P0DN78,NaN,[]


In [73]:
# stack unique canonical mapping and filtered mapping
iso_canonical_mapping = pd.concat([iso_canonical_no_duplicates, unique_mapping_df], ignore_index=True)
iso_canonical_mapping

,Entry,Canonical_Isoform_ID,Isoforms
0,A0A087X1C5,NaN,[]
1,A0A096LP01,A0A096LP01-1,[A0A096LP01-2]
2,A0A0B4J2F0,NaN,[]
3,A0A0C5B5G6,NaN,[]
4,A0A0K2S4Q6,A0A0K2S4Q6-1,[A0A0K2S4Q6-2]
...,...,...,...
20345,Q5DJT8_P0DMV1_P0DMV2,NaN,[]
20346,Q9BYR9_P0C7H8,NaN,[]
20347,A0A0A6YYL3_H3BUK9,NaN,[]
20348,A0A075B759_P0DN26,NaN,[]


In [74]:
# add unique identifier for isoform of 'A6NHP3' and 'Q495Y8'
target_entries = ['A6NHP3', 'Q495Y8']

# Use .loc to find those rows and update the 'Isoforms' column
iso_canonical_mapping.loc[iso_canonical_mapping['Entry'].isin(target_entries), 'Isoforms'] = [['A6NHP3-2_Q495Y8-2']]

In [75]:
iso_canonical_mapping[iso_canonical_mapping['Entry'] == 'A6NHP3']

,Entry,Canonical_Isoform_ID,Isoforms
18892,A6NHP3,A6NHP3-1,[A6NHP3-2_Q495Y8-2]


In [76]:
iso_canonical_mapping[iso_canonical_mapping['Entry'] == 'Q495Y8']

,Entry,Canonical_Isoform_ID,Isoforms
19299,Q495Y8,Q495Y8-1,[A6NHP3-2_Q495Y8-2]


After the digestion, we found that there are proteins in Uniprot for which isoforms are listed, that are themselves canonical forms. So we will delete these from the list of isoforms in the mapping and only keep them as individul canonical forms with their direct isoforms.

'Q9NZJ9-2', 'Q96PT3-2', 'P0DPK4-2' were already deleted from the duplicates list and replaced with their base id further up and in notebook 1.3. -> probably can be removed from this list.

'P0DPB3-1', 'P63092-1', 'P0DP91-1', and 'P0DPB6-1' are canonical forms, but have NaN values in the Canonical_Isoform_ID column in the mapping so they are still in the isoform list. I don't know why this happened, because it did not happen for any other IDs. I will therefore adjust these entries by hand. 

In [89]:
# List of IDs not found in digestion df
ids_to_check = [
    'P0DPB3-1', 'Q1A5X6-1', 'Q9NPD5-1', 'G3V0H7-1', 'Q9NQG6-1', 'Q5TFQ8-1', 'O00291-2', 'O00712-3', 
    'O14628-2', 'O14628-3', 'O15119-4', 'O15409-3', 'O15534-2', 'O15534-3', 'Q9P0M2-1', 'Q8IX05-1', 
    'B3EWF7-1', 'E9PQ53-1', 'Q5JWF2-1', 'P63092-1', 'P06881-1', 'F8WCM5-1', 'P04150-4', 'F7VJQ1-1', 
    'P01258-1', 'Q6EEV4-1', 'Q9BZG1-1', 'P0DMS9-1', 'P0DMS8-1', 'P35638-1', 'P10826-4', 'P13497-7', 
    'P20963-2', 'P26367-3', 'P30307-5', 'B1AH88-1', 'P0DPQ6-1', 'Q13948-1', 'P40855-2', 'P40855-3', 
    'P40855-4', 'P42167-1', 'P42166-1', 'P42702-2', 'Q8N726-1', 'P47756-3', 'P52272-3', 'P52757-2', 
    'Q9ULB1-1', 'Q9P2S2-1', 'Q6ZVN7-1', 'P63126-1', 'P0DW28-1', 'P0DP91-1', 'Q08493-4', 'Q08493-5', 
    'Q08493-6', 'Q08493-7', 'Q08648-6', 'Q13243-4', 'Q13621-2', 'Q13705-2', 'E9PAV3-1', 'P39880-1', 
    'Q14114-5', 'Q2M385-2', 'O95467-1', 'B7ZAP0-1', 'G9CGD6-1', 'Q9Y3L3-1', 'Q96GD0-1', 'Q70YC5-1', 
    'Q70YC4-1', 'Q8IWY9-3', 'O60449-1', 'Q5JU69-1', 'P42771-1', 'Q9H496-1', 'Q8WVD5-2', 'O95411-1', 
    'Q93009-2', 'L0R6Q1-1', 'Q6ZT62-1', 'Q96JB1-3', 'Q96JB1-4', 'Q96LB4-2', 'Q9BXH1-1', 'Q99259-2', 
    'Q96PG8-2', 'P0DI83-1', 'Q9H0M0-4', 'Q9H0M0-5', 'Q9HBH9-3', 'Q9HBH9-4', 'Q9HBH9-5', 'Q9Y4C0-1', 
    'F5H094-1', 'Q9NPG1-2', 'L0R8F8-1', 'Q9NZJ9-2', 'O43687-1', 'P58401-1', 'Q96EF9-1', 'Q9UL45-3', 
    'P58400-2', 'O94854-3', 'Q9Y281-2', 'Q9HDB5-1', 'O95278-1', 'Q13765-1', 'Q6P9H4-1', 'Q8WWN9-1', 
    'Q9UPN3-1', 'Q92614-1', 'P98175-1', 'P56278-1', 'P56277-1', 'P63128-1', 'Q8N2E6-1', 'O00241-1', 
    'Q9UKY1-1', 'Q96K31-1', 'Q9HAT1-2', 'P0C7T4-1', 'Q5R372-1', 'Q9BQ13-1', 'O95298-1', 'P04156-1', 
    'Q96G79-1', 'Q9NWL6-1', 'A8MTL9-1', 'P0DPB6-1', 'P0CAP2-1', 'Q8N7M0-2', 'Q9HC47-1', 'Q9BYG7-3', 
    'Q9BYG7-4', 'L0R819-1', 'P30536-1', 'P01308-1', 'B3KU38-1', 'Q8IU53-1', 'P60896-1', 'Q6XLA1-1', 
    'Q8NFQ8-1', 'Q96RT6-1', 'Q96PT3-2', 'P0DPK4-2'
]

# List of the 50 IDs with no sequence in UniProt
no_uniprot_seq = {
    'Q14114-5', 'Q93009-2', 'Q13243-4', 'P30307-5', 'O00712-3', 'Q08493-4', 'O14628-2', 'O14628-3', 
    'Q9HBH9-3', 'Q08493-6', 'Q08493-5', 'P47756-3', 'Q96LB4-2', 'Q96JB1-4', 'Q9NPG1-2', 'Q8IWY9-3', 
    'Q08648-6', 'Q9BYG7-3', 'P40855-2', 'P40855-3', 'O15409-3', 'P26367-3', 'P42702-2', 'Q13621-2', 
    'P52272-3', 'Q9HBH9-4', 'O15534-3', 'Q9UL45-3', 'Q99259-2', 'Q9Y281-2', 'P52757-2', 'Q9HBH9-5', 
    'P10826-4', 'O15534-2', 'O00291-2', 'Q9HAT1-2', 'Q2M385-2', 'Q96JB1-3', 'P13497-7', 'Q9H0M0-4', 
    'Q8WVD5-2', 'P20963-2', 'Q9H0M0-5', 'P40855-4', 'P04150-4', 'Q13705-2', 'O15119-4', 'Q9BYG7-4', 
    'Q08493-7', 'Q8N7M0-2'
}

# List of IDs to remove from isoform lists (not all because not all of them are canonical)
ids_keep =  {
    'Q9NZJ9-2', 'Q96PT3-2', 'P0DPK4-2'
}

# Create a new list excluding those IDs
ids_to_remove = {i for i in ids_to_check if i not in ids_keep}

iso_canonical_mapping['Isoforms'] = iso_canonical_mapping['Isoforms'].apply(
    lambda x: [item for item in x if item not in ids_to_remove]
)

In [96]:
# Use .loc to find those rows and update the 'Isoforms' column
for idx in iso_canonical_mapping.index[iso_canonical_mapping['Entry'] == "P0DPB3"]:
    iso_canonical_mapping.at[idx, 'Canonical_Isoform_ID'] = 'P0DPB3-1'
    iso_canonical_mapping.at[idx, 'Isoforms'] = ['P0DPB3-2', 'P0DPB3-3', 'P0DPB3-4']

for idx in iso_canonical_mapping.index[iso_canonical_mapping['Entry'] == "P63092"]:
    iso_canonical_mapping.at[idx, 'Canonical_Isoform_ID'] = 'P63092-1'
    iso_canonical_mapping.at[idx, 'Isoforms'] = ['P63092-2', 'P63092-3']

for idx in iso_canonical_mapping.index[iso_canonical_mapping['Entry'] == "P0DP91"]:
    iso_canonical_mapping.at[idx, 'Canonical_Isoform_ID'] = np.nan
    iso_canonical_mapping.at[idx, 'Isoforms'] = []

for idx in iso_canonical_mapping.index[iso_canonical_mapping['Entry'] == "P0DPB6"]:
    iso_canonical_mapping.at[idx, 'Canonical_Isoform_ID'] = np.nan
    iso_canonical_mapping.at[idx, 'Isoforms'] = []

In [97]:
# Create the "Flat" table
iso_canonical_mapping_flat = iso_canonical_mapping.explode('Isoforms').rename(columns={'Isoforms': 'Isoform_ID'})

In [98]:
iso_canonical_mapping_flat = (
    iso_canonical_mapping
    .explode('Isoforms')
    .rename(columns={'Isoforms': 'Isoform_ID'})
    .dropna(subset=['Isoform_ID', 'Canonical_Isoform_ID'])
)

iso_canonical_mapping_flat = iso_canonical_mapping_flat.reset_index(drop=True)
iso_canonical_mapping_flat

,Entry,Canonical_Isoform_ID,Isoform_ID
0,A0A096LP01,A0A096LP01-1,A0A096LP01-2
1,A0A0K2S4Q6,A0A0K2S4Q6-1,A0A0K2S4Q6-2
2,A0A1B0GTW7,A0A1B0GTW7-1,A0A1B0GTW7-2
3,A0A1B0GTW7,A0A1B0GTW7-1,A0A1B0GTW7-3
4,A0AV02,A0AV02-1,A0AV02-2
...,...,...,...
22183,B7ZAQ6_P0CG08,B7ZAQ6_P0CG08,B7ZAQ6-2
22184,B7ZAQ6_P0CG08,B7ZAQ6_P0CG08,B7ZAQ6-3
22185,B7ZAQ6_P0CG08,B7ZAQ6_P0CG08,B7ZAQ6-4
22186,P0DI81_P0DI82,P0DI81_P0DI82,P0DI81-2


In [99]:
# Save as csv 
iso_canonical_mapping.to_csv(os.path.join(mapping_dir, 'iso_canonical_mapping.csv'), index=False)
iso_canonical_mapping_flat.to_csv(os.path.join(mapping_dir, 'iso_canonical_mapping_flat.csv'), index=False)

In [94]:
num_false = (~iso_canonical_mapping["Entry"].isin(canonical_unique["ID"])).sum()

print(num_false)

0


In [95]:
num_false = (~canonical_unique["ID"].isin(iso_canonical_mapping["Entry"])).sum()

print(num_false)

0
